# VanaDrishti on Colab — 01: Phase 1 baseline reproduction (U-TAE)

Thin driver: every cell calls code that lives in the repo (`src/`, `scripts/`), so the notebook stays in sync with the tested code. Outputs go to `MyDrive/vanadrishti/`.

## 1. Setup (run every session)

Before running: **Runtime → Change runtime type → T4 GPU** (any GPU works).
Colab forgets everything except Google Drive when the session ends, so the code and the dataset are
re-fetched each session and all run outputs (checkpoints, logs, metrics) are written to Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/vana-drishti'
BRANCH = 'claude/new-session-uto7x0'
DRIVE = '/content/drive/MyDrive/vanadrishti'
RUNS = f'{DRIVE}/runs'
STATS = f'{DRIVE}/stats/vanadrishti_close_train_stats.pt'
!mkdir -p {RUNS} {DRIVE}/stats {DRIVE}/results

In [ ]:
import os
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q -b {BRANCH} https://github.com/krish-k1301/vana-drishti.git {REPO}
%cd {REPO}
!git log --oneline -1

Pinned versions from `environment.yml`. torch 2.5.1 is required: the upstream reference code calls
`torch.load` in a way that breaks on torch >= 2.6. pip may print dependency-conflict warnings about
other Colab packages; they don't affect this project. All training runs as `!python ...` subprocesses,
so no runtime restart is needed after the install.

In [ ]:
!pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu124
!pip install -q lightning==2.4.0 torchmetrics==1.9.0 numpy==2.1.3 pandas==2.2.3 scipy==1.17.1 pyyaml==6.0.3 einops==0.8.2 matplotlib==3.11.2 tqdm==4.70.1 requests==2.34.2 pytest==9.1.1
!python -c "import torch, lightning; print(torch.__version__, lightning.__version__, 'CUDA:', torch.cuda.is_available())"

## 2. Dataset

BraDD-S1TS from Zenodo: a 17.7 GB zip, md5-checked. The **first** session downloads it and keeps a copy on
Drive (`MyDrive/vanadrishti/data/`); every later session copies that one file back (a few minutes) instead of
downloading. The data is always **unzipped to Colab's local disk**: training reads ~26,000 small files, which
is far too slow from Drive. Skipped entirely if the data is already unzipped in this session.

In [ ]:
import glob
ZIP_NAME = 'BraDD-S1TS_zenodo.zip'
ZIP_DRIVE = f'{DRIVE}/data/{ZIP_NAME}'
ZIP_LOCAL = f'{REPO}/data/{ZIP_NAME}'

def find_root():
    hits = sorted(glob.glob(f'{REPO}/data/**/meta.csv', recursive=True))
    return os.path.dirname(hits[0]) if hits else None

if find_root() is None:
    !mkdir -p {REPO}/data {DRIVE}/data
    !df -h /content | tail -1
    if os.path.exists(ZIP_DRIVE):
        print('copying the cached zip from Drive ...')
        !cp {ZIP_DRIVE} {ZIP_LOCAL}
        !python scripts/download_bradd.py --config configs/data/downloads.yaml --delete-zip
    else:
        !python scripts/download_bradd.py --config configs/data/downloads.yaml
        print('caching the verified zip on Drive for the next sessions ...')
        !cp {ZIP_LOCAL} {ZIP_DRIVE} && rm {ZIP_LOCAL}
ROOT = find_root()
print('BraDD root:', ROOT)
assert ROOT is not None, 'meta.csv not found: check the output above'

## 3. Phase 1: baseline reproduction (PRD Section 4.2 / 7)

Four runs, every PRD 4.2 value set explicitly in the configs:

| Key | Launcher | Config | Target pixel IoU |
|---|---|---|---|
| `ref_ce` | upstream code (`src.reference_run`) | `configs/reference_baseline_utae.yaml` | 47.3 ± 2 |
| `ours_ce` | our pipeline (`src.train`) | `configs/baseline_utae.yaml` | 47.3 ± 2, within 1 of `ref_ce` |
| `ref_focal` | upstream code | `configs/reference_baseline_utae_focal.yaml` | 48.6 ± 2 |
| `ours_focal` | our pipeline | `configs/baseline_utae_focal.yaml` | 48.6 ± 2, within 1 of `ref_focal` |

Runs **resume automatically**: if Colab disconnects, run the notebook again with the same `RUN`;
training continues from the last finished epoch (checkpoints live on Drive). Max 200 epochs, early
stopping after 30 epochs without improvement, so one run usually needs several sessions.

In [ ]:
RUNS_TABLE = {
    'ref_ce':     ('src.reference_run', 'configs/reference_baseline_utae.yaml'),
    'ours_ce':    ('src.train',         'configs/baseline_utae.yaml'),
    'ref_focal':  ('src.reference_run', 'configs/reference_baseline_utae_focal.yaml'),
    'ours_focal': ('src.train',         'configs/baseline_utae_focal.yaml'),
}
RUN = 'ours_ce'
LAUNCHER, CONFIG = RUNS_TABLE[RUN]
print(RUN, LAUNCHER, CONFIG)

### Optional, first time only: time one epoch

Trains one epoch into a separate throw-away run (`timing_<RUN>`) and prints the epoch time and peak
GPU memory, so you know how many sessions a full run needs. Skip once you know.

In [ ]:
!python -m {LAUNCHER} --config {CONFIG} data.root={ROOT} data.stats_path={STATS} data.num_workers=2 output.runs_dir={RUNS} trainer.max_epochs=1 train.resume=false experiment_name=timing_{RUN}

In [ ]:
import pandas as pd
m = pd.read_csv(f'{RUNS}/timing_{RUN}/csv/version_0/metrics.csv')
print(m.filter(regex='epoch_time|peak_gpu|pixel_iou').dropna(how='all').tail())

### Full run (resumes if interrupted)

In [ ]:
!python -m {LAUNCHER} --config {CONFIG} data.root={ROOT} data.stats_path={STATS} data.num_workers=2 output.runs_dir={RUNS}

In [ ]:
import json, yaml
exp = yaml.safe_load(open(CONFIG))['experiment_name']
print(json.dumps(json.load(open(f'{RUNS}/{exp}/metrics_test.json')), indent=1))